In [ ]:
import torch
from torch.nn import functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
device='cuda'
# model_id = "meta-llama/Llama-3.2-1B"
model_id = "unsloth/Llama-3.2-1B"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.eval();

In [ ]:
text = "The capital of France is"
inputs = tokenizer(text, return_tensors="pt").to(device)

In [ ]:
#Tokenizer prepends a speical <|begin_of_text|> token
for input_token_index in inputs['input_ids'].view(-1):
    print(input_token_index.item(), tokenizer.decode(input_token_index))

In [ ]:
#Pass inputs into model
with torch.no_grad():
    outputs=model(inputs["input_ids"])

In [ ]:
#One output vector for each input token, each vector has one value for each token in Llama's 128256 token vocabulary
outputs.logits.shape 

In [ ]:
probabilities=F.softmax(outputs.logits, dim=-1) #Convert to probabilities (more on this in Ch. 3)
probabilities.shape

In [ ]:
#Just look at final vector to see what text the model predicts next
top_probs, top_indices = torch.topk(probabilities[0, -1, :], 10)
for i, (prob, idx) in enumerate(zip(top_probs, top_indices), 1):
    print(idx.item(), round(probabilities[0, -1, idx].item(),5), tokenizer.decode([idx]))

In [ ]:
tokenizer.decode([0])

In [ ]:
tokenizer.decode([1])

In [ ]:
tokenizer.decode([128255])

In [ ]:
tokenizer.decode([128254])